# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

In [3]:
!pip install tensorflow==2.15
!pip install transformers


ERROR: Could not find a version that satisfies the requirement tensorflow==2.15 (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0)
ERROR: No matching distribution found for tensorflow==2.15


### Setup

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')
from  dotenv import load_dotenv

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain_classic import hub
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool



load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  # leave room for GPT-4 response
    return trimmed

prompt = hub.pull("hwchase17/react")
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
agent = create_react_agent(llm, tools, prompt)

# LangChain tool
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
agent = AgentExecutor(tools=tools, llm=llm, agent=agent, verbose=True)

C:\Users\VORPC\AppData\Local\Temp\ipykernel_2340\4270882616.py:15: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_search = TavilySearchResults()


NameError: name 'tiktoken' is not defined

In [2]:
import os
import warnings
from dotenv import load_dotenv

from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults

import tiktoken

warnings.filterwarnings("ignore")
load_dotenv()

# ---- LLM ----
llm = ChatOpenAI(model="gpt-4", temperature=0)

# ---- Tavily Tool ----
tavily = TavilySearchResults()
encoding = tiktoken.encoding_for_model("gpt-4")

def safe_search(query: str) -> str:
    result = tavily.run(query)
    text = str(result)

    tokens = encoding.encode(text)
    return encoding.decode(tokens[:3500])

tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Search the web safely and return trimmed results"
    )
]

# ---- ReAct Prompt ----

prompt = hub.pull("hwchase17/react")

# ---- Create Agent ----
agent = create_react_agent(llm, tools, prompt)

# ---- Executor ----
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# ---- Run ----
agent_executor.invoke({"input": "Who is the CEO of OpenAI?"})




> Entering new AgentExecutor chain...
I need to find out who the current CEO of OpenAI is.
Action: TavilySafeSearch
Action Input: "CEO of OpenAI"[{'title': 'OpenAI - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/OpenAI', 'content': 'CEO and co-founder: Sam Altman, former president of the start-up accelerator Y Combinator "Y Combinator (company)")\n   President and co-founder: Greg Brockman, former CTO, 3rd employee of Stripe "Stripe (company)")(\n   Chief Scientist Officer: Jakub Pachocki, former Director of Research at OpenAI(\n   Chief Operating Officer: Brad Lightcap, previously at Y Combinator "Y Combinator (company)") and JPMorgan Chase(\n   Chief Financial Officer: Sarah Friar, former Nextdoor CEO and former CFO at Block, Inc.(\n   Chief Product Officer: Kevin Weil, previously at Twitter, Inc. and Meta Platforms(\n   Chief Research Officer: Mark Chen, former SVP of Research at OpenAI(\n   Chief Compliance Officer: Scott Schools, former Chief Compliance Officer of Uber "Uber

{'input': 'Who is the CEO of OpenAI?',
 'output': 'Sam Altman is the CEO of OpenAI.'}

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [4]:
prompt_1 = """Retrieve the most recent and credible advancements in generative AI applied to healthcare during the year 2025.
Summarize the developments in a structured response in Markdown that includes:
1. Key breakthroughs or innovations with short descriptions.
2. Concrete real-world deployments, pilots, or market growth figures.
3. Examples of research papers, technical progress, or standards/guidelines.
4. Ethical, regulatory, or clinical safety considerations.
5. Major trends shaping 2026 outlook.
Organize the output clearly under headings like:
# Breakthroughs
# Real-World Deployments
# Research & Standards
# Challenges & Ethics
# Trends to Watch
"""

response_1 = agent_executor.invoke({"input": prompt_1})
print(response_1)




> Entering new AgentExecutor chain...
I need to search for recent advancements in generative AI applied to healthcare during the year 2025. I will use TavilySafeSearch to find this information.
Action: TavilySafeSearch
Action Input: "Generative AI in healthcare advancements 2025" [{'title': 'Generative AI in Healthcare: Use Cases, Benefits, and Challenges', 'url': 'https://www.johnsnowlabs.com/generative-ai-healthcare/', 'content': 'Generative AI in healthcare in 2025 is set to redefine patient care, medical research, and operational efficiencies. Compared to 2025, when early adoption focused on pilot projects and regulatory discussions, 2026 will witness widespread implementation, more robust compliance frameworks, and advanced integration with multimodal AI models.\n\nKey Generative AI Trends in Healthcare in 2025: [...] Greater Adoption of Multimodal AI Models  \n  In 2025, most healthcare generative AI applications relied on single-modality models, primarily text-based large lang

In [6]:
from IPython.display import display, Markdown

# Get the plain text output
text_output = response_1['output']  # <- most executors store the text here

# Display as Markdown
display(Markdown(text_output))



# Breakthroughs
- Greater adoption of multimodal AI models: In 2025, most healthcare generative AI applications relied on single-modality models, primarily text-based large language models (LLMs). The shift toward multimodal models capable of analyzing and generating text, images, genomics data, and even real-time patient vitals simultaneously is expected to improve diagnostics, clinical decision support, and medical imaging analysis.
- Generative AI in diagnostics, patient engagement, and drug development: AI tools like ChatGPT are enabling more personalized care. Innovations such as Path Chat, a diagnostic tool that leverages AI to streamline medical imaging and support clinical decision-making, are noteworthy.

# Real-World Deployments
- Generative AI as a virtual assistant: Generative AI is being used to organize clinical notes and simplify the ways patient information is communicated across teams. It is also being used to summarize vast historical reports to give care teams immediate insights into a patient’s history.
- Automation of expert-intensive tasks: Generative AI is being used to augment radiology, dermatology, genetics, drug discovery, and electronic-health-record analysis.

# Research & Standards
- Research in generative AI in healthcare is focusing on data augmentation for imbalanced or privacy-restricted datasets, automation of expert-intensive tasks such as radiology reporting, and generation of new biomedical knowledge ranging from molecular scaffolds to fairness insights.

# Challenges & Ethics
- Ethical AI Governance: Stricter regulations are being put in place to ensure ethical AI deployment in healthcare.
- Privacy concerns and regulatory hurdles: As the technology advances, so do the challenges—ranging from ethical dilemmas to privacy concerns and regulatory hurdles.

# Trends to Watch
- Widespread implementation of generative AI in healthcare: 2026 will witness widespread implementation, more robust compliance frameworks, and advanced integration with multimodal AI models.
- Increased accuracy and transparency in Generative AI: Some healthcare organizations are beginning to experiment with retrieval-augmented generation (RAG), an AI framework that combines traditional vector database capabilities with LLMs. This is expected to produce better answers for staff in Q&A applications and reduce some of the pitfalls of using generative AI tools.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [ ]:
prompt_2 = """Use the search tool once to identify 5 emerging AI startups in 2025.
Summarize briefly in Markdown with:
- Company
- Innovation
- Source
Keep it concise.
"""




In [15]:
response_2 = agent_executor.invoke({"input": prompt_2})



> Entering new AgentExecutor chain...
I need to use the TavilySafeSearch tool to find information about emerging AI startups in 2025.
Action: TavilySafeSearch
Action Input: Emerging AI startups in 2025[{'title': 'AI 100: The most promising artificial intelligence startups of 2025', 'url': 'https://multiversecomputing.com/resources/ai-100-the-most-promising-artificial-intelligence-startups-of-2025', 'content': 'Aerospace & defense: AI solutions designed for aerospace engineering, aviation operations, military applications, and defense systems, including autonomous navigation and threat detection technologies. For instance, Quantum Systems creates eVTOL unmanned aerial systems that serve critical defense applications, most notably in Ukraine.\n Auto & mobility: AI applications for autonomous vehicles, transportation optimization, fleet management, and mobility services. Companies like Wayve are developing AI systems that use LLMs to provide real-time natural language explanations of dr

In [16]:
# Extract the text
text_output = response_2['output']

# Display nicely as Markdown
display(Markdown(text_output))


- **Company**: Quantum Systems
  **Innovation**: Creates eVTOL unmanned aerial systems for critical defense applications.
  **Source**: [multiversecomputing.com](https://multiversecomputing.com/resources/ai-100-the-most-promising-artificial-intelligence-startups-of-2025)

- **Company**: Wayve
  **Innovation**: Developing AI systems that use LLMs to provide real-time natural language explanations of driving decisions.
  **Source**: [multiversecomputing.com](https://multiversecomputing.com/resources/ai-100-the-most-promising-artificial-intelligence-startups-of-2025)

- **Company**: Overjet
  **Innovation**: Provides an AI platform that enhances clinical care through radiographic analysis and optimizes claims processing for providers and payers.
  **Source**: [multiversecomputing.com](https://multiversecomputing.com/resources/ai-100-the-most-promising-artificial-intelligence-startups-of-2025)

- **Company**: Liminal
  **Innovation**: Leverages ultrasound and machine learning inspection solutions to improve battery cell quality, cost-effectiveness, and safety.
  **Source**: [multiversecomputing.com](https://multiversecomputing.com/resources/ai-100-the-most-promising-artificial-intelligence-startups-of-2025)

- **Company**: Skyfire
  **Innovation**: Financial stack enables AI agents to perform transactions without credit cards or bank accounts.
  **Source**: [multiversecomputing.com](https://multiversecomputing.com/resources/ai-100-the-most-promising-artificial-intelligence-startups-of-2025)

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [24]:
prompt_3 = """Compare ChatGPT Team vs Claude Pro (2025).
List:
- Key features
- Pricing

Format in Markdown. Keep concise.(not go over tokens per min (TPM): Limit 10000))
"""




In [25]:
response_3 = agent_executor.invoke({"input": prompt_3})



> Entering new AgentExecutor chain...
I need to find information about the key features and pricing of both ChatGPT Team and Claude Pro in 2025. I will use TavilySafeSearch to find this information.
Action: TavilySafeSearch
Action Input: ChatGPT Team key features and pricing 2025[{'title': 'ChatGPT Team Plan (2025): Full Features, Costs & Comparison', 'url': 'https://www.brainchat.ai/blog/chatgpt-team-plan', 'content': 'Enhanced collaboration tools that make teamwork smoother\n Strong security features that keep your data safe\n The potential drawbacks you’ll need to consider, like pricing and flexibility\n\nLet’s dive in and explore whether ChatGPT Team is the game-changer your business needs to get ahead!\n\n\u200d\n\n## ChatGPT Teams Key Points\n\n Price: $30/user per month (minimum of 2 seats)\n Best For: Teams of 10+ employees, businesses needing enhanced collaboration tools\n Key Features: Enhanced message limits, customizable ChatGPTs, robust security and privacy\n Drawbacks: 

In [26]:

text_output = response_3["output"]

from IPython.display import display, Markdown
display(Markdown(text_output))


# ChatGPT Team vs Claude Pro (2025)

## Key Features

### ChatGPT Team
- Enhanced message limits
- Customizable ChatGPTs
- Robust security and privacy
- Access to GPT-4 with no message limits
- Shared team workspace and folders
- Admin tools for user roles and billing
- DALL·E image generation and code interpreter
- SOC 2 compliance and no data used to train OpenAI’s models
- Priority access to new tools and model upgrades

### Claude Pro
- 5× more usage than free Claude
- Priority access during peak times
- Early feature access to new capabilities
- Extended context for longer conversations
- Higher quality responses with better reasoning
- Commercial use license for business projects
- Latest model access (Claude 3.5 Sonnet, Opus)
- No degraded performance during high traffic

## Pricing

### ChatGPT Team
- $30/user per month (minimum of 2 seats)
- $25/user per month if you pay for the whole year upfront (which saves you about 17%)

### Claude Pro
- $20/month officially through Anthropic
- $17/month if billed annually ($200 upfront)

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


test